In [ ]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

In [ ]:
%autoreload
import torch
import torch.nn as nn
from torch.optim import SGD, Adam
from torch.nn import MSELoss, BCEWithLogitsLoss
from source.normal.data import trainLoader
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import ConcatDataset, DataLoader
from source.normal.model import EfficientModel
from source.normal.train import trainModel
from source.normal.loss import CustomLoss
from source.normal.optim import RAdam
from apex import amp

In [ ]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/train/train/'
    loader['label_path'] = '../../data/train/train_folds.csv'
    loader['size'] = 256
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    train = DataLoader(train, batch_size=20, shuffle=True, num_workers=6, drop_last=True)
    valid = DataLoader(valid, batch_size=2, shuffle=True, num_workers=6, drop_last=True)
    model = EfficientModel()
    weights = torch.load('../../model/pretrain/model_1.pt')['model_state_dict']
    model.load_state_dict(weights)
    model = model.to('cuda:0')
    optimizer = RAdam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    model, optimizer = amp.initialize(model, optimizer, opt_level="O2",keep_batchnorm_fp32=True, verbosity=0)
    schedular = StepLR(optimizer, step_size=5, gamma=0.1)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = CustomLoss(weight=0.75, variance=0.)
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/train/model_{}.pt'.format(fold)
    trainer['epochs'] = 12
    trainer['batch'] = 20
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    return None

In [ ]:
train(1)

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)

In [ ]:
train(5)